In [1]:
# pip install sentence-transformers pandas

In [2]:
# pip install python-docx python-dotenv

In [3]:
from docx import Document
import os

folder_path = "test_data/"   # thư mục chứa nhiều file docx

docs = []

for file_name in os.listdir(folder_path):
    if file_name.endswith(".docx"):
        file_path = os.path.join(folder_path, file_name)

        doc = Document(file_path)

        for i, p in enumerate(doc.paragraphs):
            text = p.text.strip()
            if text:
                docs.append({
                    "chunk_id": i,
                    "text": text,
                    "file_path": file_path,
                    "file_name": file_name
                })

print("✅ Tổng số đoạn:", len(docs))
print("✅ 3 đoạn đầu tiên:")
print(docs[:3])


✅ Tổng số đoạn: 5691
✅ 3 đoạn đầu tiên:
[{'chunk_id': 0, 'text': 'Họ, tên thí sinh:', 'file_path': 'test_data/01_TdtntTayNguyen_Ntt.docx', 'file_name': '01_TdtntTayNguyen_Ntt.docx'}, {'chunk_id': 1, 'text': 'Số báo danh:……………………………………………………………….', 'file_path': 'test_data/01_TdtntTayNguyen_Ntt.docx', 'file_name': '01_TdtntTayNguyen_Ntt.docx'}, {'chunk_id': 3, 'text': 'PHẦN I. Câu trắc nghiệm nhiều phương án lựa chọn. Thí sinh trả lời từ câu 1 đến câu 18. Mỗi câu hỏi thí sinh chỉ chọn một phương án trả lời.', 'file_path': 'test_data/01_TdtntTayNguyen_Ntt.docx', 'file_name': '01_TdtntTayNguyen_Ntt.docx'}]


In [4]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import json

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

rows = []

for d in docs:
    vec = embedder.encode(d["text"]).tolist()

    rows.append({
        "chunk_id": d["chunk_id"],
        "text": d["text"],
        "embedding": json.dumps(vec),
        "file_path": d["file_path"],
        "file_name": d["file_name"]
    })

df = pd.DataFrame(rows)
df.to_csv("vector_store.csv", index=False)

print("✅ Đã lưu vector_store.csv")


d:\data datn\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Đã lưu vector_store.csv


In [1]:
import pandas as pd
import json

df = pd.read_csv("vector_store.csv")

# kiểm tra độ dài vector của từng row
df["embedding_len"] = df["embedding"].apply(lambda x: len(json.loads(x)))

print(df["embedding_len"].value_counts())


embedding_len
384    5691
Name: count, dtype: int64


In [5]:
df.to_csv("vector_store.csv", index=False)
print("✅ Đã lưu vector vào file vector_store.csv")


✅ Đã lưu vector vào file vector_store.csv


In [6]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

login(os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
import pandas as pd
import ast
import numpy as np

df = pd.read_csv("vector_store.csv")
df["embedding"] = df["embedding"].apply(ast.literal_eval)

doc_embeddings = np.array(df["embedding"].tolist())
doc_texts = df["text"].tolist()


In [8]:
from numpy.linalg import norm

def search(query, k=3):
    q_emb = embedder.encode([query])[0]

    sims = doc_embeddings @ q_emb / (
        norm(doc_embeddings, axis=1) * norm(q_emb)
    )

    top_k = sims.argsort()[-k:][::-1]

    results = []
    for i in top_k:
        results.append({
            "text": df.iloc[i]["text"],
            "file_path": df.iloc[i]["file_path"],
            "file_name": df.iloc[i]["file_name"],
            "score": float(sims[i])
        })

    return results


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
def rag_answer(query):
    results = search(query, k=3)

    context = ""
    file_suggestions = []

    for r in results:
        context += r["text"] + "\n"

        file_suggestions.append({
            "file_name": r["file_name"],
            "file_path": r["file_path"],
            "score": r["score"]
        })

    prompt = f"""
Use ONLY the context below to answer.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.7
    )

    answer = tokenizer.decode(output[0], skip_special_tokens=True)

    return {
        "answer": answer,
        "files": file_suggestions
    }


In [ ]:
result = rag_answer("Tỉnh nào sau đây giáp cả Lào và Trung Quốc?")


print("✅ Answer:\n", result["answer"])
print("\n📂 File chứa câu hỏi:")

for f in result["files"]:
    print("-", f["file_name"], "=>", f["file_path"], "| score:", f["score"])